# Lesson 3.2 — Multi-AZ & Multi-Region Architectures

**Level: Intermediate** | **Exam:** SOA-C03 Domain 2 Task 2.2; SAP-C02 Domain 1 Task 1.3

## Learning Objectives
- Distinguish Multi-AZ (HA) from Multi-Region (DR) and know when each applies
- Identify services that are inherently Multi-AZ vs those requiring explicit configuration
- Configure RDS Multi-AZ and understand automatic failover behavior
- Implement Route 53 failover routing with health checks for DNS-level DR

_Reference: [RDS Multi-AZ](https://docs.aws.amazon.com/AmazonRDS/latest/UserGuide/Concepts.MultiAZ.html) | [Route 53 Health Checks](https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/dns-failover.html)_

## Multi-AZ vs Multi-Region

| | Multi-AZ | Multi-Region |
|--|---------|-------------|
| **Protects against** | AZ failure, hardware failure | Region failure, catastrophic events |
| **Latency** | < 2ms between AZs | 50–300ms between regions |
| **Cost** | Low incremental (2x for standby) | High (duplicate infra + data transfer) |
| **Failover** | Automatic (minutes) | DNS-based (minutes to hours) |
| **Data** | Synchronous replication | Asynchronous replication |

## Services: Inherently Multi-AZ (no config needed)
S3, DynamoDB (standard), SQS, SNS, Lambda, ELB, CloudFront, Route 53

## Services: Multi-AZ Requires Configuration
- **RDS**: `MultiAZ=True` → synchronous standby in another AZ
- **ElastiCache Redis**: Cluster mode or Multi-AZ replication group
- **EC2**: Deploy with ASG across multiple AZs
- **EFS**: Automatically replicates across AZs in a Region

## RDS Multi-AZ Behavior
```
Writes → Primary (AZ-a) ──synchronous──→ Standby (AZ-b)
Reads  → Primary only (standby not readable!)
Failover → DNS points to standby (~1-2 min), old primary becomes new standby
```

## Route 53 Routing Policies Cheat Sheet
| Policy | Use Case |
|--------|----------|
| **Failover** | Active/Passive DR — send to secondary when primary fails |
| **Weighted** | A/B testing, traffic shifting (e.g., 90/10) |
| **Latency** | Route users to lowest-latency region |
| **Geolocation** | Route by user country/continent |
| **Geoproximity** | Route with bias adjustment for geographic control |
| **Multivalue** | Up to 8 healthy IPs returned (client-side load balancing) |

In [ ]:
import boto3, json

rds = boto3.client('rds', region_name='us-east-1')
route53 = boto3.client('route53')
ec2 = boto3.client('ec2', region_name='us-east-1')
print('Setup complete.')

In [ ]:
# Audit RDS instances for Multi-AZ configuration
instances = rds.describe_db_instances()['DBInstances']

if instances:
    print(f'RDS Audit ({len(instances)} instances):')
    for db in instances:
        multi_az = 'OK' if db['MultiAZ'] else 'RISK: Not Multi-AZ!'
        backup = 'OK' if db['BackupRetentionPeriod'] > 0 else 'RISK: No backups!'
        encrypted = 'OK' if db['StorageEncrypted'] else 'RISK: Not encrypted!'
        print(f'\n  DB: {db["DBInstanceIdentifier"]}')
        print(f'    Multi-AZ   : {multi_az}')
        print(f'    Backup     : {backup} (retention: {db["BackupRetentionPeriod"]} days)')
        print(f'    Encryption : {encrypted}')
else:
    print('No RDS instances found.')
    print('\nRDS Multi-AZ creation CLI example:')
    print('aws rds create-db-instance \\')
    print('  --db-instance-identifier myapp-prod \\')
    print('  --db-instance-class db.t3.medium \\')
    print('  --engine postgres \\')
    print('  --multi-az \\           # <-- This enables Multi-AZ')
    print('  --storage-encrypted')

In [ ]:
# Design a Route 53 Multi-Region Failover architecture

architecture = {
    'domain': 'myapp.example.com',
    'strategy': 'Active/Passive Failover',
    'primary': {
        'region': 'us-east-1',
        'record': {'type': 'A ALIAS', 'target': 'us-east-alb.us-east-1.elb.amazonaws.com'},
        'health_check': {'protocol': 'HTTPS', 'path': '/health', 'interval': 30},
        'failover_role': 'PRIMARY'
    },
    'secondary': {
        'region': 'eu-west-1',
        'record': {'type': 'A ALIAS', 'target': 'eu-west-alb.eu-west-1.elb.amazonaws.com'},
        'health_check': None,   # Secondary doesn't need health check
        'failover_role': 'SECONDARY'
    },
    'behavior': [
        'Normally: All traffic → us-east-1 (PRIMARY)',
        'Health check fails × 3: Route 53 → eu-west-1 (SECONDARY)',
        'Primary recovers: Route 53 → us-east-1 (after TTL expires)'
    ]
}

print('Multi-Region Failover Architecture:')
print(f'  Domain: {architecture["domain"]}')
print(f'  Strategy: {architecture["strategy"]}')
print(f'\n  Primary: {architecture["primary"]["region"]}')
print(f'    Record: {architecture["primary"]["record"]}')
print(f'    Health check: {architecture["primary"]["health_check"]}')
print(f'\n  Secondary: {architecture["secondary"]["region"]}')
print(f'    Record: {architecture["secondary"]["record"]}')
print(f'\n  Behavior:')
for b in architecture['behavior']:
    print(f'    {b}')

## Exercise

In [ ]:
# TODO: For each service, state whether Multi-AZ is automatic or requires configuration,
# and describe the appropriate setup approach

services = [
    {'service': 'Amazon RDS PostgreSQL', 'automatic': 'TODO', 'setup': 'TODO'},
    {'service': 'Amazon DynamoDB', 'automatic': 'TODO', 'setup': 'TODO'},
    {'service': 'Amazon S3', 'automatic': 'TODO', 'setup': 'TODO'},
    {'service': 'Amazon ElastiCache Redis', 'automatic': 'TODO', 'setup': 'TODO'},
    {'service': 'AWS Lambda', 'automatic': 'TODO', 'setup': 'TODO'},
    {'service': 'Amazon EC2', 'automatic': 'TODO', 'setup': 'TODO'},
]

for s in services:
    print(f"Service: {s['service']}")
    print(f"  Automatic Multi-AZ: {s['automatic']}")
    print(f"  Setup: {s['setup']}")
    print()

## Summary
- Multi-AZ = **high availability** within a region; Multi-Region = **disaster recovery**
- RDS Multi-AZ standby is **NOT** a read replica — it's hot standby only
- Route 53 failover routing + health checks provides **DNS-level DR** with automatic switchover
- DynamoDB Global Tables = multi-region **active-active** (different from RDS Multi-AZ active-passive)

---
**Next:** [Lesson 3.3 — Disaster Recovery Patterns (RTO/RPO)](lesson-3.3-disaster-recovery.ipynb)